In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
from ml_experiments.analyze import get_df_runs_from_mlflow_sql, get_missing_entries, get_common_combinations, get_df_with_combinations
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
import os
import pickle

# Save Results

## Load mlflow runs

In [ ]:
results_dir = Path.cwd().parent / "results" / "scale_features"
os.makedirs(results_dir, exist_ok=True)

In [ ]:
db_port = 5001
db_name = 'cohirf'
url = f'postgresql://user@localhost:{db_port}/{db_name}'
engine = create_engine(url)
query = 'SELECT experiments.name from experiments'
experiment_names = pd.read_sql(query, engine)['name'].tolist()


In [ ]:
experiment_names

In [ ]:
experiments_names = [exp for exp in experiment_names if exp.startswith('sfni')]

In [ ]:
experiments_names

In [ ]:
params_columns = [
    # "model_nickname",
    "model",
    "seed_model",
    "seed_dataset",
    "n_samples",
    "n_random",
    "n_informative",
    "n_features_dataset",
    "pct_random",
    "class_sep",
    # "seed_unified",
    "n_classes",
    "best/child_run_id",
]

In [ ]:
latest_metrics_columns = [
    "fit_model_return_elapsed_time",
    # "best_n_clusters_",
    # "best_rand_score",
    # "best_adjusted_rand",
    # "best_mutual_info",
    # "best_adjusted_mutual_info",
    # "best_normalized_mutual_info",
    # "best_homogeneity_completeness_v_measure",
    # "best_silhouette",
    # "best_calinski_harabasz_score",
    # "best_davies_bouldin_score",
    # "best_inertia_score",
    # "best_homogeneity",
    # "best_completeness",
    # "best_v_measure",
    "best/n_clusters_",
    "best/rand_score",
    "best/adjusted_rand",
    "best/mutual_info",
    "best/adjusted_mutual_info",
    "best/normalized_mutual_info",
    "best/homogeneity_completeness_v_measure",
    "best/silhouette",
    "best/calinski_harabasz_score",
    # "best/davies_bouldin_score",
    # "best/inertia_score",
    "best/homogeneity",
    "best/completeness",
    "best/v_measure",
	"best/elapsed_time",
]

In [ ]:
tags_columns = [
    'raised_exception',
    'EXCEPTION',
    'mlflow.parentRunId',
    # 'best_child_run_id',
]

In [ ]:
runs_columns = ['run_uuid', 'status', 'start_time', 'end_time']
experiments_columns = []
other_table = 'params'
other_table_keys = params_columns
df_params = get_df_runs_from_mlflow_sql(engine, runs_columns=runs_columns, experiments_columns=experiments_columns, experiments_names=experiments_names, other_table=other_table, other_table_keys=other_table_keys)
df_latest_metrics = get_df_runs_from_mlflow_sql(engine, runs_columns=['run_uuid'], experiments_columns=experiments_columns, experiments_names=experiments_names, other_table='latest_metrics', other_table_keys=latest_metrics_columns)
df_tags = get_df_runs_from_mlflow_sql(engine, runs_columns=['run_uuid'], experiments_columns=experiments_columns, experiments_names=experiments_names, other_table='tags', other_table_keys=tags_columns)

In [ ]:
df_runs_raw = df_params.join(df_latest_metrics)
df_runs_raw = df_runs_raw.join(df_tags)


In [ ]:
df_runs_raw.to_csv(results_dir / 'df_runs_raw.csv', index=True)

In [ ]:
df_runs_raw = pd.read_csv(results_dir / "df_runs_raw.csv", index_col=0)
df_runs_raw_parents = df_runs_raw.copy()
df_runs_raw_parents = df_runs_raw_parents.loc[df_runs_raw_parents['mlflow.parentRunId'].isna()]

In [ ]:
df_runs_raw_parents

## Delete duplicate runs (if any) and complete some models that cannot run with some datasets

In [ ]:
non_duplicate_columns = [
    # "model_nickname",
	"model",
    "seed_model",
    "seed_dataset",
    "n_samples",
    "n_random",
    "n_informative",
    "n_features_dataset",
    "pct_random",
    "class_sep",
    # "seed_unified",
    "n_classes",
]
df_runs_parents = df_runs_raw_parents.dropna(axis=0, how="all", subset=["best/adjusted_rand"]).copy()
df_runs_parents = df_runs_parents.loc[(~df_runs_parents.duplicated(non_duplicate_columns))]
# fill missing values with "None"
df_runs_parents = df_runs_parents.fillna("None")

# Missing

In [ ]:
model_nickname = df_runs_parents['model'].unique().tolist()
model_nickname.sort()
model_nickname

In [ ]:
model_nickname = [
    "CoHiRF",
    "KMeans",
    "CoHiRF-1000",
    "CoHiRF-top-down",
    "CoHiRF-top-down-inv",
]

In [ ]:
non_duplicate_columns = [
    "model",
    "n_random",
    "seed_dataset",
	"n_informative",
	"n_samples",
    "n_classes",
]

In [ ]:
n_informative = [3]
n_samples = [1000]
n_classes = [5]
n_random = [10, 100, 1000, 10000]
seed_dataset = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, n_random, seed_dataset, n_informative, n_samples, n_classes]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)
# df_missing = df_missing.rename(columns={"seed_unified": "seeds_unified"})

In [ ]:
df_missing

In [ ]:
# Join df_runs_raw_parents into df_missing using non_duplicate_columns to get the EXCEPTION column
df_missing_with_exception = df_missing.merge(
    df_runs_raw_parents[non_duplicate_columns + ["raised_exception"]],
    how="left",
    left_on=["model", "n_random", "seed_dataset", "n_informative"],
    right_on=["model", "n_random", "seed_dataset", "n_informative"],
)
df_missing_with_exception[["model", "n_random", "seed_dataset", "n_informative", "raised_exception"]]

In [ ]:
df_missing["hpo_seed"] = df_missing["seed_dataset"]
df_missing["class_sep"] = 3*(df_missing["n_informative"]**0.5)
df_missing.to_csv(results_dir / "df_missing.csv")

# Plots

In [ ]:
def plot_results(df):
    plt.style.use("seaborn-v0_8-whitegrid")
    with mpl.rc_context(
        rc={
            "font.family": "serif",
            "font.serif": ["DejaVu Serif", "Liberation Serif", "Times", "serif"],  # Reliable serif fonts
            "mathtext.fontset": "dejavuserif",  # Use DejaVu for math text to avoid missing glyphs
            "axes.unicode_minus": False,  # Use ASCII minus instead of Unicode minus
            "font.size": 12,
            "axes.linewidth": 1.2,
            "axes.labelsize": 16,
            "axes.titlesize": 16,
            "xtick.labelsize": 16,
            "ytick.labelsize": 16,
            "legend.fontsize": 10,
            "legend.frameon": True,
            "legend.fancybox": False,
            "legend.shadow": False,
            "legend.framealpha": 0.9,
            "legend.edgecolor": "black",
            "grid.alpha": 0.5,
            "axes.grid": True,
            "grid.linewidth": 0.5,
        }
    ):
        cm = 1 / 2.54  # centimeters to inches
        scale = 3.0
        fig, axs = plt.subplots(2, 1, sharex=True, figsize=(12 * scale * cm, 7 * scale * cm))
        axs = axs.flatten()
        palette = sns.color_palette("tab10", n_colors=len(df["Model"].unique()))
        # Time plot
        ax = sns.lineplot(
            data=df,
            x="Number of Non Informative Features",
            y="Time (s)",
            hue="Model",
            style="Model",
            markers=True,
            markersize=10,
            # dashes=False,
            errorbar="ci",
            ax=axs[0],
            palette=palette,
        )
        ax.set_yscale("log")
        ax.set_ylabel("Time (s)")
        ax.grid(True, which="both", linestyle="--", linewidth=0.5)
        # ARI plot
        ax2 = sns.lineplot(
            data=df,
            x="Number of Non Informative Features",
            y="ARI",
            hue="Model",
            style="Model",
            markers=True,
            markersize=10,
            # dashes=False,
            errorbar="ci",
            ax=axs[1],
            palette=palette,
        )
        ax2.set_ylabel("Adjusted Rand Index (ARI)")
        ax2.grid(True, which="both", linestyle="--", linewidth=0.5)
        ax2.set_xscale("log")
        ax.tick_params(direction="in", length=4, width=1.2)
        ax2.tick_params(direction="in", length=4, width=1.2)
        for spine in ["left", "right", "top", "bottom"]:
            ax.spines[spine].set_linewidth(1.4)
            ax.spines[spine].set_color("black")
            ax2.spines[spine].set_linewidth(1.4)
            ax2.spines[spine].set_color("black")
        # Remove duplicate legends
        handles, labels = ax.get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper center", ncol=4, fontsize=13, frameon=True, bbox_to_anchor=(0.5, 1.04))
        ax.get_legend().remove()
        ax2.get_legend().remove()
        plt.xlabel("Number of Non Informative Features")
        plt.tight_layout()
        return fig, axs

## HPO Time

In [ ]:
df = df_runs_parents.copy()
n_informative = 3
n_samples = 1000
n_classes = 5
class_sep = 3*(n_informative**0.5)
models_names = {
    "CoHiRF": "CoHiRF",
    "CoHiRF-top-down": "R-CoHiRF",
    "KMeans": "K-Means",
}
df = df.loc[
    (df["n_informative"] == n_informative)
    & (df["n_samples"] == n_samples)
    & (df["n_classes"] == n_classes)
    & (df["class_sep"] == class_sep)
    & (df["model"].isin(models_names.keys()))
]
df = df.replace({"model": models_names})
df = df.sort_values(by="model")
df = df.rename(
    columns={
        "fit_model_return_elapsed_time": "Time (s)",
        "max_memory_used": "Memory (MB)",
        "n_samples": "Number of samples",
        "n_random": "Number of Non Informative Features",
        "model": "Model",
        "best/adjusted_rand": "ARI",
    }
)

fig, axs = plot_results(df)
fig.savefig(
    results_dir
    / f"scale_features_non_informatives_results_n_samples_{n_samples}_n_classes_{n_classes}_n_informative_{n_informative}_class_sep_{class_sep}.pdf",
    dpi=300,
)
plt.show()

## Best Time

In [ ]:
df = df_runs_parents.copy()
n_informative = 3
n_samples = 1000
n_classes = 5
class_sep = 3 * (n_informative**0.5)
models_names = {
    "CoHiRF": "CoHiRF",
    "CoHiRF-top-down": "R-CoHiRF",
    "KMeans": "K-Means",
}
df = df.loc[
    (df["n_informative"] == n_informative)
    & (df["n_samples"] == n_samples)
    & (df["n_classes"] == n_classes)
    & (df["class_sep"] == class_sep)
    & (df["model"].isin(models_names.keys()))
]
df = df.replace({"model": models_names})
df = df.sort_values(by="model")
df = df.rename(
    columns={
        "best/elapsed_time": "Time (s)",
        "max_memory_used": "Memory (MB)",
        "n_samples": "Number of samples",
        "n_random": "Number of Non Informative Features",
        "model": "Model",
        "best/adjusted_rand": "ARI",
    }
)

fig, axs = plot_results(df)
fig.savefig(
    results_dir
    / f"best_scale_features_non_informatives_results_n_samples_{n_samples}_n_classes_{n_classes}_n_informative_{n_informative}_class_sep_{class_sep}.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()